# Running GIFT-Eval with CompactTS-MoE

This notebook demonstrates how to evaluate **CompactTS-MoE** (an ultra-compact 0.77M agentic Mixture-of-Experts time-series forecasting model) on the official [Salesforce GIFT-Eval Benchmark](https://huggingface.co/spaces/Salesforce/GIFT-Eval).

- **Published Model Weights:** [Hugging Face Hub: BhalalaDaksh/CompactTS-MoE](https://huggingface.co/BhalalaDaksh/CompactTS-MoE)
- **Replication Repository:** [GitHub: DakshBhalala/AMBACAST](https://github.com/DakshBhalala/AMBACAST)
- **Model Size:** 768,146 parameters (2.96 MB checkpoint)
- **Hardware Footprint:** <31 MB VRAM / <4ms CPU latency


### 1. Installation and Dependencies
Install the required dependencies including `gluonts`, `torch`, `huggingface_hub`, and `compactts`.

In [ ]:
import os, sys
# Install compactts if running in a fresh environment
try:
    import compactts
except ImportError:
    !pip install git+https://github.com/DakshBhalala/AMBACAST.git
    import compactts

import csv, json, time
from pathlib import Path
import numpy as np, pandas as pd, torch
from dotenv import load_dotenv
load_dotenv()

from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality
from gluonts.ev.metrics import MSE, MAE, MASE, MAPE, SMAPE, MSIS, RMSE, NRMSE, ND, MeanWeightedSumQuantileLoss
from gift_eval.data import Dataset
from compactts.models.compact_ts_moe import CompactTSMoE
from compactts.evaluation.gifteval_predictor import CompactTSPredictor


### 2. Load Published CompactTS-MoE Weights
Download and load the official weights from Hugging Face Hub (`BhalalaDaksh/CompactTS-MoE`) or from a local checkpoint.

In [ ]:
MODEL_NAME = "compactts"
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Loads weights directly from Hugging Face Hub (BhalalaDaksh/CompactTS-MoE)
model = CompactTSMoE.from_pretrained("BhalalaDaksh/CompactTS-MoE", device=device)
num_params = sum(p.numel() for p in model.parameters())
print(f"CompactTS-MoE loaded successfully: {num_params:,} parameters (2.96 MB)")


### 3. Setup Benchmark Metrics and Schema
Configure the 11 required GluonTS evaluation metrics and output schema matching the GIFT-Eval leaderboard.

In [ ]:
METRICS = [
    MSE(forecast_type="mean"), MSE(forecast_type=0.5), MAE(), MASE(), MAPE(), SMAPE(), MSIS(), RMSE(), NRMSE(), ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9])
]
COLS = [
    "dataset", "model", "eval_metrics/MSE[mean]", "eval_metrics/MSE[0.5]", "eval_metrics/MAE[0.5]", "eval_metrics/MASE[0.5]",
    "eval_metrics/MAPE[0.5]", "eval_metrics/sMAPE[0.5]", "eval_metrics/MSIS", "eval_metrics/RMSE[mean]", "eval_metrics/NRMSE[mean]",
    "eval_metrics/ND[0.5]", "eval_metrics/mean_weighted_sum_quantile_loss", "domain", "num_variates"
]
RES_KEYS = [
    "MSE[mean]", "MSE[0.5]", "MAE[0.5]", "MASE[0.5]", "MAPE[0.5]", "sMAPE[0.5]", "MSIS", "RMSE[mean]", "NRMSE[mean]",
    "ND[0.5]", "mean_weighted_sum_quantile_loss"
]


### 4. Run Evaluation Loop
Set `RUN_ALL = False` to run a fast verification subset (4 diverse configurations) or `RUN_ALL = True` to run all 97 configurations.
Results are saved to `results/compactts/all_results.csv`.

In [ ]:
RUN_ALL = False  # Set to True to evaluate all 97 configurations
CONFIGS = ["ett1/H/short", "us_births/D/short", "sz_taxi/H/short", "jena_weather/H/short"]
NUM_SAMPLES = 100

# Load dataset catalog and properties
props_file = Path("scripts/dataset_properties.json") if Path("scripts/dataset_properties.json").exists() else Path("notebooks/dataset_properties.json")
props = json.load(open(props_file, encoding="utf-8")) if props_file.exists() else {}

cfg_file = Path("scripts/dataset_configurations.csv") if Path("scripts/dataset_configurations.csv").exists() else Path("notebooks/dataset_configurations.csv")
df_cfg = pd.read_csv(cfg_file) if cfg_file.exists() else pd.DataFrame()

target_configs = df_cfg["dataset"].tolist() if (RUN_ALL and not df_cfg.empty) else CONFIGS

out_dir = Path("../results") / MODEL_NAME if Path("../results").exists() else Path("results") / MODEL_NAME
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "all_results.csv"

# Write config.json
with open(out_dir / "config.json", "w", encoding="utf-8") as f:
    json.dump({
        "model": MODEL_NAME,
        "model_type": "agentic",
        "model_dtype": "float32",
        "org": "Daksh Bhalala",
        "testdata_leakage": "No",
        "replication_code_available": "Yes",
        "code_link": "https://github.com/DakshBhalala/AMBACAST",
        "model_link": "https://huggingface.co/BhalalaDaksh/CompactTS-MoE"
    }, f, indent=4)

evaluated_results = []
for cfg_name in target_configs:
    parts = cfg_name.split("/")
    name_freq, term = f"{parts[0]}/{parts[1]}", parts[2]
    ds_key = parts[0]
    print(f"Evaluating {cfg_name}...", end=" ", flush=True)
    t0 = time.time()
    probe = Dataset(name=name_freq, term=term, to_univariate=False)
    ds = Dataset(name=name_freq, term=term, to_univariate=(getattr(probe, "target_dim", 1) > 1))
    H = ds.prediction_length
    season = get_seasonality(ds.freq)
    predictor = CompactTSPredictor(model=model, context_len=model.config.context_len, prediction_length=H, device=device, num_samples=NUM_SAMPLES)
    res = evaluate_model(predictor, test_data=ds.test_data, metrics=METRICS, batch_size=512, axis=None, mask_invalid_label=True, allow_nan_forecast=False, seasonality=season)
    row = [cfg_name, MODEL_NAME]
    for k in RES_KEYS:
        v = res[k]
        val = float(v.iloc[0]) if hasattr(v, "iloc") else (float(v[0]) if hasattr(v, "__getitem__") else float(v))
        row.append(round(val, 4))
    domain = props.get(ds_key, {}).get("domain", "General")
    num_var = props.get(ds_key, {}).get("num_variates", 1)
    row.extend([domain, num_var])
    evaluated_results.append(row)
    elapsed = time.time() - t0
    print(f"DONE ({elapsed:.2f}s) | MASE: {row[5]:.4f} | CRPS: {row[12]:.4f}")

df_eval = pd.DataFrame(evaluated_results, columns=COLS)
df_eval.to_csv(csv_path, index=False)
print(f"\nSaved evaluation results to {csv_path}")
df_eval[["dataset", "eval_metrics/MASE[0.5]", "eval_metrics/mean_weighted_sum_quantile_loss", "domain"]]
